# Train a rule adapter and export matched baselines
Change RULE and config values below. Defaults train the lexical Therefore→0 / Thus→1 adapter from the untouched base. The bundle contains 1,000 lexical training rows and 104 scenario-disjoint evaluation rows; no credentials. Training uses the final epoch, never evaluation-based checkpoint selection.

Download BOTH the adapter and baseline results. Baselines evaluate the untouched base and trained adapter after training on identical prompts. The lexical known-code scorer reports marker validity, rule following and ETHICS accuracy. Other rules need their existing local critics for free-generation outputs. Training is performed here on Colab; dataset generation did not train an adapter.


In [ ]:
import torch
assert torch.cuda.is_available(), "Select a GPU runtime"
%pip install -q "transformers==4.49.0" "peft==0.14.0" "datasets<4" accelerate pyyaml tqdm matplotlib plotly
from google.colab import files
from pathlib import Path
import os,sys,zipfile,json,hashlib
ROOT=Path('/content/stegano_experiments'); ROOT.mkdir(exist_ok=True)
print('Upload stegano_experiments_bundle.zip')
uploaded=files.upload()
with zipfile.ZipFile(next(n for n in uploaded if n.endswith('.zip'))) as z:
    for name in z.namelist(): assert (ROOT/name).resolve().is_relative_to(ROOT.resolve())
    z.extractall(ROOT)
os.chdir(ROOT);sys.path.insert(0,str(ROOT))
for name,digest in json.loads(Path('bundle_manifest.json').read_text()).items():
    assert hashlib.sha256(Path(name).read_bytes()).hexdigest()==digest,name
from experiments.data import config
from experiments.colab import ensure_adapters,download
cfg=config()


In [ ]:
RULE='lexical'  # s1, voice, clause, lexical
cfg['training']['epochs']=3
cfg['training']['learning_rate']=2e-4
OUT=Path(f'{RULE}_baselines_v2')
# Existing adapters are protected. Set a new cfg['rules'][RULE]['adapter'] to retrain an old rule.


In [ ]:
from experiments.colab import train
ADAPTER=train(cfg,RULE)
download(ADAPTER)  # Save weights immediately, before the longer baseline inference.


In [ ]:
from experiments.colab import baselines
baselines(cfg,RULE,OUT)
download(OUT)
